# GEMM_V2：CPU、PL 单核/双核/三核公平对比

统一测试 INT8 `2048×2048 @ 2048×2048`，输出 INT32。CPU 使用四线程 NEON；PL 的 A/B 准备、B 转置也使用四线程 NEON。每种路径预热后测 6 轮，固定输入，校验 PL 输出与 CPU 参考逐元素一致。

PL 端到端时间包含普通主存 A→CMA、B 转置到 CMA、缓存同步、软件配置/启动/等待、结果同步及 C→独立普通主存。分开报告端到端 GOPS 与 PL 硬件周期换算 GOPS。乘法和累加各计 1 次操作，所以工作量为 `2×M×N×K`。

CPUFreq 会请求板上最高可用档位（1199999 kHz，约 1.2 GHz 档），同时记录 CPU 实际频率。当前板卡读数约 1049.99 MHz；若读回低于请求值，Notebook 会明确显示，不会把设定档位当成实际频率。

**PL 频率设置为 250 MHz。仅运行 GEMM，不运行 Softmax。**

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

BASE = Path.cwd()
assert BASE.name == "GEMM", f"请在 GEMM 目录运行，此处为 {BASE}"
sys.path.insert(0, str(BASE))
from benchmark_gemm_2048 import main

# 设置最高可用 CPUFreq 档位，使用四线程 NEON；每种配置跑 6 轮。
results = main(repeats=6)
OPS_G = 2 * 2048 * 2048 * 2048 / 1e9
print(f"矩阵乘法工作量：{OPS_G:.6f} GOP（乘加按 2 OP 计）")
print(f"CPU 实际读回：{results['cpu_mhz']:.4f} MHz；PL 实际读回：{results['clock_mhz']:.4f} MHz")
print("CPU：4 线程 NEON；B 转置：4 线程 NEON；PL GEMM 核：1 / 2 / 3 个")

## 吞吐与端到端时间

`端到端 GOPS = 工作量 / total_s`；PL `硬件 GOPS = 工作量 / 最慢核心周期 / PL_MHz`。PL 的软件提交时间、硬件等待时间分别列出；等待阶段与 PL 硬件运行重叠，不能把两项相加后再加回总时间。

In [ ]:
rows = []
for mode in ["CPU", "PL1", "PL2", "PL3"]:
    s = results["summary"][mode]
    total_s = s["total_s"]["median"]
    row = {
        "路径": {"CPU":"CPU 4T NEON", "PL1":"PL 单核", "PL2":"PL 双核", "PL3":"PL 三核"}[mode],
        "端到端中位数 (ms)": total_s * 1000,
        "端到端 (GOPS)": OPS_G / total_s,
        "实际 CPU (MHz)": results["cpu_mhz"],
        "PL (MHz)": results["clock_mhz"],
    }
    if mode == "CPU":
        compute_s = pd.Series([r["profile"]["compute_seconds"] for r in results["records"] if r["mode"] == mode]).median()
        row["CPU计算中位数 (ms)"] = compute_s * 1000
        row["CPU计算 (GOPS)"] = OPS_G / compute_s
        row["PL硬件时间 (ms)"] = None
        row["PL硬件 (GOPS)"] = None
        row["PL软件提交 (ms)"] = None
    else:
        hw_s = s["hardware_max_s"]["median"]
        row["PL硬件时间 (ms)"] = hw_s * 1000
        row["PL硬件 (GOPS)"] = OPS_G / hw_s
        row["PL软件提交 (ms)"] = s["submit_s"]["median"] * 1000
        row["PL等待阶段 (ms) [含硬件]"] = s["wait_s"]["median"] * 1000
        row["端到端减硬件 (ms)"] = (total_s - hw_s) * 1000
    rows.append(row)
summary_table = pd.DataFrame(rows).set_index("路径")
display(summary_table.round(3))
print("CPUFreq 最高请求档位：", results["cpu_frequency"])

## 软件与搬运分项

PL 每轮的软件/数据准备和输出时间，均是端到端计时内部的分项。`PL等待阶段`包含硬件执行；`端到端减硬件`是端到端总时间减去硬件周期换算时间的剩余项，不是独立可并行相加的硬件阶段。

In [ ]:
breakdown = []
for mode in ["CPU", "PL1", "PL2", "PL3"]:
    records = [r for r in results["records"] if r["mode"] == mode]
    if mode == "CPU":
        def med(path): return pd.Series([path(r) for r in records]).median() * 1000
        breakdown.append({
            "路径":"CPU 4T NEON",
            "CPU打包/转置 (ms)": med(lambda r:r["profile"]["packing_seconds"]),
            "CPU GEMM计算 (ms)": med(lambda r:r["profile"]["compute_seconds"]),
            "A复制 (ms)": 0.0, "PL B转置 (ms)": 0.0,
            "缓存同步 (ms)": 0.0, "PL软件提交 (ms)": 0.0,
            "PL等待阶段 (ms)": 0.0,
            "输出复制 (ms)": med(lambda r:r["output_copy_s"]),
            "总时间 (ms)": med(lambda r:r["total_s"]),
        })
    else:
        def med(key): return pd.Series([r[key] for r in records]).median() * 1000
        breakdown.append({
            "路径":mode,
            "CPU打包/转置 (ms)":0.0, "CPU GEMM计算 (ms)":0.0,
            "A复制 (ms)":med("copy_a_s"),
            "PL B转置 (ms)":med("transpose_call_s"),
            "缓存同步 (ms)":med("input_sync_s")+med("output_sync_s"),
            "PL软件提交 (ms)":med("submit_s"),
            "PL等待阶段 (ms)":med("wait_s"),
            "输出复制 (ms)":med("output_copy_s"),
            "总时间 (ms)":med("total_s"),
        })
breakdown_table = pd.DataFrame(breakdown).set_index("路径")
display(breakdown_table.round(3))

## 每个 PL 核的工作量与周期

三核将 2048 行分成 64 个 32 行块，预期为 704/672/672 行。每核周期换算时间使用同一 PL 时钟；双核/三核的硬件关键路径取参与核心中周期最多的那个核。

In [ ]:
core_rows = []
for mode in ["PL1", "PL2", "PL3"]:
    records = [r for r in results["records"] if r["mode"] == mode]
    for core in records[0]["cores"]:
        parts = [p for r in records for p in r["rows"] if p["core"] == core]
        cycles = pd.Series([p["counters"]["cycles"] for p in parts]).median()
        compute_cycles = pd.Series([p["counters"]["compute_cycles"] for p in parts]).median()
        core_rows.append({
            "配置":mode, "核心":core,
            "行起始":parts[-1]["start"], "行数":parts[-1]["rows"],
            "总周期中位数":int(cycles), "计算周期中位数":int(compute_cycles),
            "该核周期折算 (ms)":cycles/(results["clock_mhz"]*1e3),
        })
core_table = pd.DataFrame(core_rows).set_index(["配置","核心"])
display(core_table.round(3))
assert results["all_verified"]
assert all(r["verified"] for r in results["records"])
assert all(not p["counters"]["saturated"] for r in results["records"] if r["mode"] != "CPU" for p in r["rows"])
assert [(r["start"],r["rows"]) for r in next(r for r in results["records"] if r["mode"]=="PL3")["rows"]] == [(0,704),(704,672),(1376,672)]
print("通过：CPU / PL 单核 / 双核 / 三核输出全部逐元素一致；行分块对齐 32；硬件周期计数未饱和。")
print("JSON 结果：", sorted((BASE/"benchmark_results").glob("gemm_2048_*.json"))[-1])